# Projeto Semantix — Preparação dos Dados

## Objetivo

Nesta etapa, os dados serão preparados para as etapas posteriores de modelagem e avaliação.

As decisões de tratamento serão baseadas nos problemas identificados durante a análise exploratória, buscando preservar o máximo possível das informações disponíveis e evitar alterações desnecessárias nos dados.

## Principais pontos identificados na exploração

- Valores ausentes em `person_emp_length` e `loan_int_rate`;
- Valores extremos em `person_age`, `person_income` e `person_emp_length`;
- Variáveis categóricas que precisarão ser transformadas para utilização nos modelos;
- Distribuição desigual da variável-alvo `loan_status`.

## Etapas da preparação

1. Tratamento dos valores ausentes;
2. Tratamento dos valores extremos;
3. Transformação das variáveis categóricas;
4. Separação entre variáveis preditoras e variável-alvo;
5. Divisão dos dados em treino e teste;
6. Verificação final da base preparada.

# Bibliotecas - 

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [27]:
# Carregando dataset
df = pd.read_csv('../data/credit_risk_dataset.csv', sep=',')

df.head(5)

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


# 1. Tratamento dos valores nulos 

In [28]:
# Mediana do tempo de emprego
df["person_emp_length"].median()

4.0

In [29]:
# Mediana da taxa de juros
df["loan_int_rate"].median()

10.99

In [30]:
# Preencher valores ausentes com a mediana
df["person_emp_length"] = df["person_emp_length"].fillna(df["person_emp_length"].median())
df["loan_int_rate"] = df["loan_int_rate"].fillna(df["loan_int_rate"].median())

df.isnull().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

Os valores ausentes de person_emp_length e loan_int_rate foram preenchidos com suas respectivas medianas, preservando os registros sem introduzir valores excessivamente influenciados por extremos.

# 2. Transformação das variáveis em numéricas

In [31]:
# Verificar as categorias das variáveis categóricas
for coluna in df.select_dtypes(include="object").columns:
    print(f"\n{coluna}:")
    print(df[coluna].unique())


person_home_ownership:
['RENT' 'OWN' 'MORTGAGE' 'OTHER']

loan_intent:
['PERSONAL' 'EDUCATION' 'MEDICAL' 'VENTURE' 'HOMEIMPROVEMENT'
 'DEBTCONSOLIDATION']

loan_grade:
['D' 'B' 'C' 'A' 'E' 'F' 'G']

cb_person_default_on_file:
['Y' 'N']


In [32]:
# Transformar a classificação A-G em valores numéricos
df["loan_grade"] = df["loan_grade"].map({
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4,
    "F": 5,
    "G": 6
})

df["loan_grade"].value_counts().sort_index()

loan_grade
0    10777
1    10451
2     6458
3     3626
4      964
5      241
6       64
Name: count, dtype: int64

In [33]:
# Transformar person_home_ownership e loan_intent com One-Hot Encoding
df = pd.get_dummies(
    df,
    columns=["person_home_ownership", "loan_intent"],
    dtype=int
)

# Transformar cb_person_default_on_file em 0 e 1
df["cb_person_default_on_file"] = df["cb_person_default_on_file"].map({
    "N": 0,
    "Y": 1
})

df.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_grade',
       'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length',
       'person_home_ownership_MORTGAGE', 'person_home_ownership_OTHER',
       'person_home_ownership_OWN', 'person_home_ownership_RENT',
       'loan_intent_DEBTCONSOLIDATION', 'loan_intent_EDUCATION',
       'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
       'loan_intent_PERSONAL', 'loan_intent_VENTURE'],
      dtype='object')

As variáveis categóricas foram convertidas para formatos numéricos adequados aos modelos: One-Hot Encoding para categorias sem ordem, codificação ordinal para loan_grade e 0/1 para a variável binária.

# Tratando possíveis Outliers

In [34]:
# Verificar idades acima de 100 anos
df[df["person_age"] > 100]["person_age"].value_counts()

person_age
144    3
123    2
Name: count, dtype: int64

In [35]:
df = df[df["person_age"] <= 100]

In [36]:
# Verificar rendas acima de 1 milhão
df[df["person_income"] > 1_000_000]["person_income"].value_counts()

person_income
1200000    3
2039784    1
1362000    1
1440000    1
1782000    1
1900000    1
Name: count, dtype: int64

In [37]:
# Verificar tempos de emprego acima de 50 anos
df[df["person_emp_length"] > 50]["person_emp_length"].value_counts()

person_emp_length
123.0    2
Name: count, dtype: int64

In [38]:
# Verificar idades acima de 100 anos
df[df["person_age"] > 100]["person_age"].value_counts()

Series([], Name: count, dtype: int64)

In [39]:
# Remover registros com tempo de emprego acima de 50 anos
df = df[df["person_emp_length"] <= 50]

Foram removidos valores incompatíveis com o contexto, como idades acima de 100 anos e tempos de emprego de 123 anos. Rendas superiores a 1 milhão foram mantidas por poderem representar situações financeiras legítimas.

# 3. Separação de treino e teste

In [40]:
# Separar variáveis preditoras e variável-alvo
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

# Dividir os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [41]:
# Verificar o tamanho dos conjuntos
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (26059, 19)
X_test: (6515, 19)
y_train: (26059,)
y_test: (6515,)


In [42]:
# Salvar os conjuntos preparados
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)